# 07 Cost Awareness

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Understand how Databricks charges (DBUs × price, plus cloud infrastructure on classic compute), estimate the cost of a workload, find where money goes with the <code>system.billing</code> tables, see how data scanned drives cost, and build habits that keep bills low: right compute type, auto-termination, less data scanned, and fewer reruns.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Performance work is cost work: on Databricks you pay for compute time, so a job that runs twice as long costs twice as much. Teams are increasingly asked to explain and reduce their spend, and "how would you reduce the cost of this pipeline?" is a common senior interview question. Knowing where cost comes from lets you answer with numbers.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 How Databricks billing works</b>
<ul><li><b>DBU (Databricks Unit)</b>: a normalized unit of processing per hour. Each compute type consumes DBUs at a rate (DBUs per hour), and each <b>SKU</b> (jobs compute, all-purpose compute, SQL warehouse, serverless, model serving ...) has its own <b>price per DBU</b></li><li><b>Classic compute</b>: you pay Databricks for DBUs <b>and</b> your cloud provider for the VMs, disks and network</li><li><b>Serverless</b>: one DBU price that includes the infrastructure. You pay only while work runs (no idle cluster), and Databricks handles sizing</li><li>Cost of a run ≈ DBUs per hour × hours × price per DBU (+ VM cost on classic)</li><li>Usage is recorded in <b>system tables</b>: <code>system.billing.usage</code> (what was consumed, by which job, warehouse, notebook or user) and <code>system.billing.list_prices</code> (price per SKU over time)</li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A team's monthly bill doubled. <code>system.billing.usage</code> grouped by <code>sku_name</code> and <code>usage_metadata.job_id</code> showed that 60% of the spend came from one job running on <b>all-purpose</b> compute (an interactive cluster left attached to a scheduled job) and from a dev cluster with no auto-termination running all weekend. Moving the job to job compute and setting 30-minute auto-termination cut the bill by 45%, without touching any code.
</div>

## Cost levers at a glance

| Lever | Why it saves money |
|---|---|
| Job compute (or serverless jobs) instead of all-purpose for scheduled work | Lower DBU price, and the cluster exists only for the run |
| Auto-termination on interactive clusters | No paying for idle time |
| Serverless for bursty or short workloads | No idle time, no startup waste |
| Right-size: smaller or fewer nodes, autoscaling | Pay for what you use |
| Photon for SQL and DataFrame-heavy work | Higher DBU rate, but often much shorter runs |
| Read less data: filters, column pruning, clustering, partition pruning | Less compute time per query |
| Incremental processing (Auto Loader, streaming, MERGE on changes) | Process only new data, not everything every day |
| Table maintenance (`OPTIMIZE`, predictive optimization) | Faster reads for everyone downstream |
| Fewer failed runs and reruns | Every failed attempt is paid for |
| Tags on compute and jobs | Attribute cost to teams and projects |

## 1. Estimate the cost of a workload

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Defines a small cost estimator and compares scenarios. The prices below are <b>illustrative placeholders</b>, not real list prices: check your cloud and region's pricing page, or <code>system.billing.list_prices</code>.<br><br>
<b>Why it matters</b><br>Simple arithmetic is often enough to make a decision: whether to schedule hourly or daily, or whether a 2x speed-up from Photon pays for its higher DBU rate.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A table of scenarios with monthly estimates. The daily schedule is far cheaper than hourly, and the faster Photon run costs less even with the higher DBU rate.
</div>

In [0]:
# Illustrative numbers only: replace with your own SKU prices and measured run times.
PRICE_PER_DBU = {"jobs": 0.15, "all_purpose": 0.55, "jobs_photon": 0.15}
DBU_PER_HOUR = {"jobs": 8.0, "all_purpose": 8.0, "jobs_photon": 16.0}    # Photon consumes DBUs at a higher rate

def monthly_cost(sku, minutes_per_run, runs_per_day, vm_cost_per_hour=0.0):
    hours = minutes_per_run / 60 * runs_per_day * 30
    return hours * (DBU_PER_HOUR[sku] * PRICE_PER_DBU[sku] + vm_cost_per_hour)

scenarios = [
    ("Hourly on all-purpose cluster", "all_purpose", 20, 24),
    ("Hourly on job compute", "jobs", 20, 24),
    ("Daily on job compute", "jobs", 25, 1),
    ("Daily on job compute + Photon (2.5x faster)", "jobs_photon", 10, 1),
]
print(f"{'scenario':<46}{'monthly estimate':>18}")
for name, sku, minutes, runs in scenarios:
    print(f"{name:<46}{monthly_cost(sku, minutes, runs):>18,.2f}")

## 2. See your usage in `system.billing.usage`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Queries DBU consumption for the last 30 days by day and SKU.<br><br>
<b>Why it matters</b><br>This is the source of truth for "where does our money go?". Account or workspace admins can grant access to system tables.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Free Edition is free of charge, so billing system tables may be unavailable or empty there. The cell catches the error and prints a message. On a paid workspace with system tables enabled, it returns your real usage.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
On a paid workspace: DBUs per day and SKU. On Free Edition: probably the "not available" message.
</div>

In [0]:
try:
    display(spark.sql("""
        SELECT usage_date, sku_name, ROUND(SUM(usage_quantity), 2) AS dbus
        FROM system.billing.usage
        WHERE usage_date >= current_date() - INTERVAL 30 DAYS
        GROUP BY usage_date, sku_name
        ORDER BY usage_date DESC, dbus DESC
    """))
except Exception as e:
    print("system.billing.usage not available here:", type(e).__name__)

## 3. Estimated cost per job and warehouse

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins usage to the list prices that were valid at the time of use, and ranks the most expensive jobs and SQL warehouses.<br><br>
<b>Why it matters</b><br>The top few items usually explain most of the bill, so start optimizing there. <code>usage_metadata</code> tells you <b>what</b> consumed the DBUs: <code>job_id</code>, <code>warehouse_id</code>, <code>notebook_id</code>, <code>cluster_id</code> and more.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
On a paid workspace: the top 10 cost drivers with estimated list-price cost. Otherwise a message.
</div>

In [0]:
try:
    display(spark.sql("""
        SELECT
          COALESCE(u.usage_metadata.job_id, u.usage_metadata.warehouse_id, u.usage_metadata.cluster_id, 'other') AS consumer,
          u.sku_name,
          ROUND(SUM(u.usage_quantity), 2)                          AS dbus,
          ROUND(SUM(u.usage_quantity * lp.pricing.default), 2)     AS est_list_cost
        FROM system.billing.usage u
        JOIN system.billing.list_prices lp
          ON u.sku_name = lp.sku_name
         AND u.usage_unit = lp.usage_unit
         AND u.usage_end_time >= lp.price_start_time
         AND (lp.price_end_time IS NULL OR u.usage_end_time < lp.price_end_time)
        WHERE u.usage_date >= current_date() - INTERVAL 30 DAYS
        GROUP BY ALL
        ORDER BY est_list_cost DESC
        LIMIT 10
    """))
except Exception as e:
    print("billing system tables not available here:", type(e).__name__)

## 4. Data scanned drives compute time

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes a table partitioned by month, then counts the files a query must read with no filter, with a partition filter, and with a filter on a non-partition column.<br><br>
<b>Why it matters</b><br>Compute time (and so cost) grows with the data read. Queries that prune files by partition, clustering or data skipping statistics cost less every time they run, which matters most for dashboards and frequently scheduled jobs.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The full scan reads all files, the month filter reads 1/12 of them, and the filter on a non-partition column still has to consider all files (unless data skipping statistics or clustering help).
</div>

In [0]:
DB = "workspace.optimization"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

In [0]:
from pyspark.sql import functions as F

t = f"{DB}.sales_by_month"
(spark.range(0, 600_000, 1, numPartitions=4)
 .select("id", (F.col("id") % 12 + 1).alias("month"), (F.col("id") % 97).alias("store"), (F.col("id") % 500).alias("amount"))
 .write.mode("overwrite").partitionBy("month").saveAsTable(t))

sales = spark.table(t)
for label, df in [("no filter", sales), ("month = 3 (partition column)", sales.filter("month = 3")),
                  ("store = 5 (not a partition column)", sales.filter("store = 5"))]:
    print(f"{label:<38} files to read: {len(df.inputFiles())}")

## 5. Don't pay twice: idempotent, incremental jobs

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows the difference between reprocessing everything every day and processing only new data, by counting the rows each approach touches over a week.<br><br>
<b>Why it matters</b><br>Full reprocessing cost grows with the size of the table every day. Incremental processing cost grows only with the new data. Over months, the difference is often 10–100x.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Full reprocessing touches far more rows over the week than incremental processing.
</div>

In [0]:
existing, daily_new = 10_000_000, 200_000
full = sum(existing + daily_new * d for d in range(1, 8))
incremental = daily_new * 7
print(f"rows processed in a week, full reprocessing: {full:,}")
print(f"rows processed in a week, incremental:       {incremental:,}  ({full / incremental:.0f}x less)")

## Under the hood

```
cost of a run = (DBU/hour of the compute) × (hours it runs) × (price per DBU of the SKU)  [+ VM cost on classic]
                    └ node type, size, Photon     └ data volume, code efficiency, idle time, retries
system.billing.usage ──join on sku_name + time──▶ system.billing.list_prices  ⇒  estimated list cost
```

Everything in folders 02–05 (pruning, fewer shuffles, no skew, no UDFs) reduces the **hours** term.

In [0]:
sales.filter("month = 3").explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: scheduled jobs running on all-purpose clusters</b><br>
All-purpose compute has a higher DBU price. Use job compute or serverless for scheduled work.<br><br>
<b>⛔ Problem: dev clusters running overnight</b><br>Set auto-termination (for example 30–60 minutes) and cluster policies that enforce it.<br><br>
<b>⛔ Problem: "Photon costs more, so turn it off"</b><br>Photon has a higher DBU rate, but if it makes the job 2–3x faster, the total cost is lower. Measure the total, not the rate.<br><br>
<b>⛔ Problem: full reloads every day</b><br>Switch to incremental loading (Auto Loader, <code>MERGE</code> on changes, streaming with <code>availableNow</code>).<br><br>
<b>⛔ Problem: can't tell which team spends what</b><br>Add custom tags to compute and jobs, and group <code>system.billing.usage</code> by <code>custom_tags</code>.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is a DBU?</b><br>
A Databricks Unit: a normalized unit of processing capacity per hour. Each compute type consumes DBUs at a rate, and each SKU (jobs, all-purpose, SQL, serverless) has its own price per DBU. On classic compute you also pay the cloud provider for the VMs.<br><br>

<b>🎤 2. How would you reduce the cost of a Databricks pipeline?</b><br>
First measure with <code>system.billing.usage</code> and the query profiles to find the top consumers. Then: run scheduled work on job compute or serverless instead of all-purpose, set auto-termination, right-size or autoscale clusters, process incrementally instead of full reloads, read less data (filters, clustering, pruning), remove waste in code (UDFs, skew, extra shuffles), avoid failed reruns, and use Photon where it shortens runs enough.<br><br>

<b>🎤 3. How do you find which jobs cost the most?</b><br>
Query <code>system.billing.usage</code> grouped by <code>usage_metadata.job_id</code> (or <code>warehouse_id</code>, <code>cluster_id</code>) and join to <code>system.billing.list_prices</code> for estimated list cost, then rank them.<br><br>

<b>🎤 4. Why is serverless often cheaper even with a higher DBU price?</b><br>
There's no idle time or startup waste: you pay only while work runs, and the infrastructure is included. For bursty, short or interactive workloads, that usually outweighs the rate difference.<br><br>

<b>🎤 5. How do tags help with cost?</b><br>
Custom tags on clusters, jobs, warehouses and serverless budget policies appear in the billing usage records, so spend can be attributed to teams, projects or environments.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A nightly ETL job runs on an all-purpose cluster that stays on all day. Which change reduces cost most directly?</b><br>
A. Running the job on job compute (or serverless) so compute exists only during the run<br>
B. Adding more workers to the all-purpose cluster<br>
C. Disabling Delta Lake<br>
D. Running the job every hour instead<br>
<details><summary><b>Show answer</b></summary><b>A.</b></details><br><br>

<b>Q2. Which system table records DBU consumption by workspace, SKU and the job or warehouse that used it?</b><br>
A. <code>system.access.audit</code><br>
B. <code>system.billing.usage</code><br>
C. <code>system.compute.clusters</code><br>
D. <code>information_schema.tables</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>With <code>monthly_cost</code>, find the run time at which job compute + Photon becomes cheaper than job compute without Photon for a 30-minute job</li><li>(Paid workspace) Write a query that returns DBUs per day for the last 14 days for SQL warehouses only (<code>sku_name</code> containing <code>SQL</code>)</li><li>Using <code>inputFiles()</code>, compare the files read for <code>month IN (1, 2)</code> and <code>month &gt; 10</code></li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Photon breaks even when its run time is at most (rate without / rate with) of the original
base_cost = monthly_cost("jobs", 30, 1)
for minutes in range(30, 0, -1):
    if monthly_cost("jobs_photon", minutes, 1) <= base_cost:
        print(f"Photon is cheaper or equal if the 30-minute job runs in {minutes} minutes or less")
        break

# 2. Query (run on a paid workspace)
sql_dbus = """
SELECT usage_date, ROUND(SUM(usage_quantity), 2) AS dbus
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 14 DAYS AND sku_name LIKE '%SQL%'
GROUP BY usage_date ORDER BY usage_date
"""
print(sql_dbus)

# 3. Files read with partition filters
print("month IN (1, 2):", len(sales.filter("month IN (1, 2)").inputFiles()))
print("month > 10:     ", len(sales.filter("month > 10").inputFiles()))

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Cost ≈ DBU rate × hours × price per DBU (+ VMs on classic). Faster code = cheaper runs</li><li>Job compute or serverless for scheduled work, auto-termination for interactive clusters</li><li>Measure with <code>system.billing.usage</code> + <code>system.billing.list_prices</code>, and fix the top consumers first</li><li>Read less data, process incrementally, avoid reruns, tag everything</li><li>Judge Photon and bigger clusters by <b>total</b> cost, not rate</li></ul>
</div>

| Question | Where to look |
|---|---|
| How much did we use? | `system.billing.usage` |
| What did it cost (list price)? | Join `system.billing.list_prices` |
| Which job or warehouse? | `usage_metadata.job_id`, `.warehouse_id` |
| Which team? | `custom_tags` |
| Why is a query expensive? | Query profile: rows and bytes read, spill, time |

## Git commit

```
git add 05_optimization/07_cost_awareness.ipynb
git commit -m "add 05_07 cost awareness notebook"
```